# 06 — Product Research & Vendor Landscape

Analyzes the solution registry and classifies products into lifecycle categories.
Generates vendor landscape visualizations and continent-level summaries.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.research.product_researcher import analyze_registry, summarize_registry

%matplotlib inline

In [ ]:
df = analyze_registry()
print(f"Registry loaded: {len(df)} solutions, {df['vendor'].nunique()} unique vendors")
print(f"Continents: {df['continent'].unique().tolist()}")
print(f"Countries: {df['country_code'].unique().tolist()}")

In [ ]:
summary = summarize_registry(df)
print("Continent-level summary:")
display(summary)

## 6.1 Vendor Landscape — Solutions per Category per Continent

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
cat_order = ['cutting_edge', 'mature_active', 'most_used_current', 'most_used_eol']
palette = {'cutting_edge': '#2ecc71', 'mature_active': '#3498db',
           'most_used_current': '#f39c12', 'most_used_eol': '#e74c3c'}

ct = pd.crosstab(df['continent'], df['lifecycle_assigned'])
ct = ct[cat_order]
ct.plot(kind='bar', ax=ax, color=[palette[c] for c in ct.columns], width=0.75)
ax.set_title('Solutions per Category per Continent', fontsize=14, fontweight='bold')
ax.set_xlabel('Continent')
ax.set_ylabel('Number of Solutions')
ax.legend(title='Lifecycle Category')
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('data/processed/vendor_landscape.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved \u2192 data/processed/vendor_landscape.png")

## 6.2 Lifecycle Distribution — Donut Chart

In [ ]:
lifecycle_counts = df['lifecycle_assigned'].value_counts()
lifecycle_counts = lifecycle_counts.reindex(cat_order)
colors = [palette[c] for c in lifecycle_counts.index]

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    lifecycle_counts.values,
    labels=lifecycle_counts.index,
    autopct='%1.1f%%',
    startangle=90,
    colors=colors,
    wedgeprops=dict(width=0.4, edgecolor='w', linewidth=2),
    pctdistance=0.78,
)
ax.set_title('Lifecycle Category Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('data/processed/lifecycle_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved \u2192 data/processed/lifecycle_distribution.png")

## 6.3 Vendor Timeline — Category Presence by Country (Heatmap)

In [ ]:
heat = df.pivot_table(
    index='country_code',
    columns='lifecycle_assigned',
    values='name',
    aggfunc='count',
    fill_value=0,
)
heat = heat[cat_order]

fig, ax = plt.subplots(figsize=(14, max(6, len(heat) * 0.4)))
sns.heatmap(heat, annot=True, fmt='d', cmap='YlGnBu', linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Number of Solutions'})
ax.set_title('Lifecycle Categories by Country', fontsize=14, fontweight='bold')
ax.set_xlabel('Lifecycle Category')
ax.set_ylabel('Country Code')
plt.tight_layout()
plt.savefig('data/processed/vendor_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved \u2192 data/processed/vendor_timeline.png")

## 6.4 Continent Summary Tables

In [ ]:
for continent in df['continent'].unique():
    sub = df[df['continent'] == continent]
    print()
    print('=' * 60)
    print(f"  {continent.upper()}")
    print('=' * 60)
    display(sub.groupby('lifecycle_assigned').agg(
        count=('name', 'count'),
        vendors=('vendor', lambda x: sorted(x.unique())),
    ))

## 6.5 Key Statistics

In [ ]:
total = len(df)
per_category = df['lifecycle_assigned'].value_counts()
unique_vendors = df['vendor'].nunique()
per_continent = df.groupby('continent')['name'].count()
per_country = df.groupby('country_code')['name'].count()
per_vendor = df.groupby('vendor')['name'].count().sort_values(ascending=False)

sep = '=' * 50
print(sep)
print('  KEY STATISTICS')
print(sep)
print(f"Total solutions:        {total}")
print(f"Unique vendors:         {unique_vendors}")
print(f"Unique countries:       {df['country_code'].nunique()}")
print(f"Unique continents:      {df['continent'].nunique()}")
print()
print('--- Solutions per Lifecycle Category ---')
for cat in cat_order:
    count = per_category.get(cat, 0)
    pct = 100 * count / total if total else 0
    print(f"  {cat:25s}: {count:3d} ({pct:5.1f}%)")
print()
print('--- Solutions per Continent ---')
for cont, count in per_continent.items():
    print(f"  {cont:15s}: {count}")
print()
print('--- Solutions per Country (top 10) ---')
for cc, count in per_country.sort_values(ascending=False).head(10).items():
    print(f"  {cc:5s}: {count}")
print()
print('--- Top Vendors by Solution Count (top 10) ---')
for vendor, count in per_vendor.head(10).items():
    print(f"  {vendor:25s}: {count}")

**EN — Key statistics.** These counts summarise the vendor/solution registry by category, lifecycle and continent. Use them as denominators when reading the catalog in the frontend: e.g. how many cutting-edge cloud options exist per region.

**繁中 — 關鍵統計。** 這些計數依類別、生命週期與洲別彙整供應商/方案註冊表。閱讀前端目錄時可作為分母參考：例如各區域有多少前沿雲端方案。

## 6.6 Save Registry CSV

In [ ]:
out = Path('data/processed/solution_registry.csv')
df.to_csv(out, index=False)
print(f"Registry saved \u2192 {out.resolve()}")